# MiniMind (64M) trained from scratch on a free Kaggle GPU

Pretraining on `pretrain_t2t_mini.jsonl`, then full SFT on `sft_t2t_mini.jsonl`, with the upstream
[MiniMind](https://github.com/jingyaogong/minimind) scripts pinned at commit `7a9137d`, unmodified.

**Kaggle settings**: Accelerator **GPU T4 x2** (a single T4 works, about twice as slow), **Internet on**
(needs a phone-verified account). Run with **Save Version → Save & Run All (Commit)**: it runs in the
background, up to 12 h. Each stage stops itself after `STAGE_TIMEOUT_S`; upstream saves the weights every
1,000 steps, so a stopped stage still leaves usable weights.

**Outputs** (`/kaggle/working/minimind-out/`): `pretrain_768.pth`, `full_sft_768.pth` (fp16 weights,
about 130 MB each), `pretrain.log`, `full_sft.log` (each line prefixed with a Unix timestamp, same format
as the lab's `results/cpu-smoke.log`), `env.txt`.

Upstream reference: one epoch of each stage takes about 1.21 h + 1.10 h on one RTX 3090 (`README_en.md`,
Training Cost).

In [ ]:
import os, subprocess, sys, time
from pathlib import Path

UPSTREAM_COMMIT = "7a9137d2e90294df80ce9178b89e82657e19f5a7"
STAGE_TIMEOUT_S = int(5.5 * 3600)  # two stages must fit in Kaggle's 12 h session
WORK = Path("/kaggle/working/minimind-out")
REPO = Path("/tmp/minimind")  # scratch, not kept as output
WORK.mkdir(parents=True, exist_ok=True)
REPO.parent.mkdir(parents=True, exist_ok=True)

subprocess.run(["nvidia-smi"], check=True)
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/jingyaogong/minimind.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "-q", UPSTREAM_COMMIT], check=True)
print(subprocess.run(["git", "-C", str(REPO), "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# Upstream pins these (requirements.txt); torch with CUDA is already on the Kaggle image.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.6", "datasets==3.6.0"], check=True)
import torch, transformers, datasets
GPUS = torch.cuda.device_count()
env = f"torch {torch.__version__} cuda {torch.version.cuda} transformers {transformers.__version__} datasets {datasets.__version__} gpus {GPUS} {torch.cuda.get_device_name(0)}"
(WORK / "env.txt").write_text(env + "\n")
print(env)

In [ ]:
from huggingface_hub import hf_hub_download

# curl failed the TLS handshake with HF's Xet CDN on Kaggle (exit 35); huggingface_hub uses
# Python's TLS stack, handles the Xet backend and retries on its own.
for name in ("pretrain_t2t_mini.jsonl", "sft_t2t_mini.jsonl"):
    dest = REPO / "dataset" / name
    if not dest.exists() or dest.stat().st_size == 0:
        dest.unlink(missing_ok=True)
        hf_hub_download("jingyaogong/minimind_dataset", name, repo_type="dataset", local_dir=REPO / "dataset")
    print(name, round(dest.stat().st_size / 1e9, 2), "GB")

In [ ]:
def run_stage(script: str, log_name: str, *args: str) -> int:
    """Run an upstream trainer with torchrun, prefix each output line with a Unix timestamp."""
    cmd = ["timeout", "--signal=INT", str(STAGE_TIMEOUT_S),
           "torchrun", f"--nproc_per_node={GPUS}", script,
           "--epochs", "1", "--dtype", "float16", "--num_workers", "4", *args]
    print(" ".join(cmd))
    with open(WORK / log_name, "w") as log:
        proc = subprocess.Popen(cmd, cwd=REPO / "trainer", stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            stamped = f"{time.time():.3f} {line}"
            log.write(stamped)
            log.flush()
            if "loss" in line or "Error" in line or "Params" in line:
                print(stamped, end="")
    print("exit code", proc.wait())  # 124 = stopped by STAGE_TIMEOUT_S; last saved weights are kept
    return proc.returncode

In [ ]:
# T4 has no fast bfloat16: float16 + upstream GradScaler. Other hyperparameters: upstream defaults.
run_stage("train_pretrain.py", "pretrain.log")

In [ ]:
# Full SFT starts from ../out/pretrain_768.pth (upstream init_model reads ../out).
run_stage("train_full_sft.py", "full_sft.log", "--from_weight", "pretrain")

In [ ]:
import re, shutil
for pth in sorted((REPO / "out").glob("*.pth")):
    shutil.copy2(pth, WORK / pth.name)
line = re.compile(r"^(\d+\.\d+) Epoch:\[\d+/\d+\]\((\d+)/(\d+)\), loss: ([\d.]+)")
for log_name in ("pretrain.log", "full_sft.log"):
    pts = [m.groups() for m in map(line.match, (WORK / log_name).read_text().splitlines()) if m]
    if len(pts) > 1:
        (t0, s0, _, l0), (t1, s1, n, l1) = pts[0], pts[-1]
        print(f"{log_name}: steps {s1}/{n}, loss {float(l0):.3f} -> {float(l1):.3f}, "
              f"{(int(s1) - int(s0)) / (float(t1) - float(t0)):.2f} it/s, {(float(t1) - float(t0)) / 3600:.2f} h")
for f in sorted(WORK.iterdir()):
    print(f.name, round(f.stat().st_size / 1e6, 1), "MB")